In [0]:
#################################################
#################Oficcers#######################
#################################################
from pyspark.sql.functions import regexp_extract, col, explode, current_timestamp
from delta.tables import DeltaTable

def scd_merge_table(spark, source_df, target_table, business_key):
    if not spark.catalog.tableExists(target_table):
        source_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)
    else:
        merge_condition = " AND ".join([f"target.{c} = source.{c}" for c in business_key])
        DeltaTable.forName(spark, target_table).alias("target") \
            .merge(source_df.alias("source"), merge_condition) \
            .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

# Read bronze
df = spark.read.table("investment_intelligence_platform.bronze.officers")

# Extract company number and explode items directly
df_silver = (
    df
    .withColumn("company_number", regexp_extract(col("meta_data"), r'/([^/]+)\.json$', 1))
    .withColumn("officer", explode(col("items")))
    .select(
        "company_number",
        col("officer.name").alias("name"),
        col("officer.officer_role").alias("officer_role"),
        col("officer.appointed_on").alias("appointed_on"),
        col("officer.appointed_before").alias("appointed_before"),
        col("officer.resigned_on").alias("resigned_on"),
        col("officer.nationality").alias("nationality"),
        col("officer.country_of_residence").alias("country_of_residence"),
        col("officer.is_pre_1992_appointment").alias("is_pre_1992_appointment"),
        col("officer.person_number").alias("person_number"),
        col("officer.date_of_birth.month").alias("dob_month"),
        col("officer.date_of_birth.year").alias("dob_year"),
        col("officer.address.address_line_1").alias("address_line_1"),
        col("officer.address.locality").alias("locality"),
        col("officer.address.postal_code").alias("postal_code"),
        col("officer.address.country").alias("country"),
        col("last_updated_ts"),
        current_timestamp().alias("silver_updated_ts")
    )
)

# Save with SCD1 merge
target_table = "investment_intelligence_platform.silver.officers"
business_key = ["company_number", "person_number", "officer_role", "appointed_on"]

scd_merge_table(spark, df_silver, target_table, business_key)
print("Silver officers table complete!")
